In [38]:
import sys
import os
import numpy as np
import xarray as xr
import pandas as pd
import datetime
import fnmatch

import matplotlib as mpl
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [39]:
folder = '/Volumes/blue_wd/ESA_F4R/tropomi2/'
folder_out = '/Volumes/blue_wd/ESA_F4R/tropomi2_merged/' 
Y = 2024

In [40]:
#H2O column conversion

#molec/cm**2

#cm**2 --> kg
#(pressure in Pa)*( area in m**2)/(gravity in m/s**2)
P = 101300 #should this be 101325? 
A = 10**(-4)
g = 9.81 

conv_denom = P*A/g

#molec --> g
#(number of molecules)*(atomic weight in g/mol)/(avogadro number in molec/mol)
AW = 18 #should be 18.015?
AV = 6.02214076*10**23


In [41]:
xls = []
lon_bnds, lat_bnds = (8, 32), (12,-15)
arr = fnmatch.filter(os.listdir(folder), '*'+str(Y)+'*')
print(arr)
for I,F in enumerate(arr):
    try:
        x = xr.open_dataset(folder+F,
                            group="PRODUCT",
                            drop_variables=['level','corner','glintflag','ground_pixel','nwin',
                                            'hdo_column_precision','hdo_column_apriori',
                                            'hdo_profile_apriori','h2o_profile_apriori',
                                            'delta_time','layer',
                                            'h2o_column_precision','h2o_column_apriori',
                                            'deltad_precision','hdo_column','h2o_vmr'])
        x = x.rename({'latitude':'lat','longitude':'lon'})
        x = x.assign_coords({"ground_pixel": x.ground_pixel.values})
        x = x.where((x.lat < 12.) & (x.lat > -15.),drop=True)
        x = x.where((x.lon < 31.) & (x.lon > 14.),drop=True)
        x = x.where(x['qa_value']>=0.7,drop=True)
        x['deltad'] = x['deltad']*1000.0
        N = x['h2o_column']
        conv_enum = N*AW/AV
        x['h2o_vmr'] = conv_enum/conv_denom
#        print(x['time.year'].max())
        if x['time.year'].max().values==Y:
            print('appending!')
            xls.append(x)
        x.close()
        print(F)
    except:
        print('not nc4: ',F)

['S5P_PAL__L2__HDO__S_20180525T092024_20180525T110154_03178_01_100300_20250308T155225.nc', 'S5P_PAL__L2__HDO__S_20180712T092024_20180712T110154_03859_01_100300_20250308T211336.nc', 'S5P_PAL__L2__HDO__S_20180713T090120_20180713T104250_03873_01_100300_20250308T202411.nc', 'S5P_PAL__L2__HDO__S_20181023T102117_20181023T120247_05321_01_100300_20250308T151449.nc', 'S5P_PAL__L2__HDO__S_20181023T120247_20181023T134416_05322_01_100300_20250308T151917.nc', 'S5P_PAL__L2__HDO__S_20190706T121137_20190706T135306_08954_01_100300_20250309T120245.nc', 'S5P_PAL__L2__HDO__S_20190818T102116_20190818T120246_09563_01_100300_20250308T091354.nc', 'S5P_PAL__L2__HDO__S_20190818T120246_20190818T134415_09564_01_100300_20250308T092025.nc', 'S5P_PAL__L2__HDO__S_20201128T121803_20201128T135933_16204_01_100300_20250310T020246.nc', 'S5P_PAL__L2__HDO__S_20210319T092024_20210319T110154_17777_01_100300_20250312T025322.nc', 'S5P_PAL__L2__HDO__S_20210426T123449_20210426T141619_18318_01_100300_20250310T120241.nc', 'S5P_PAL_

In [42]:
trp = xr.concat(xls,dim='time')
trp = trp.sortby('time')
#print(trp)
trp = trp.sel(time=slice(str(Y)+'-01-01',str(Y)+'-12-31'))
trp = trp.drop_vars(['qa_value','h2o_column'])
print(trp)

trp.to_netcdf(folder_out+"TROPOMI_merged_"+str(Y)+".nc",engine='h5netcdf')



<xarray.Dataset> Size: 3GB
Dimensions:       (time: 595, scanline: 1664, ground_pixel: 215)
Coordinates:
  * time          (time) datetime64[ns] 5kB 2024-01-01T11:29:33 ... 2024-12-3...
  * scanline      (scanline) int32 7kB 1196 1197 1198 1199 ... 2857 2858 2859
  * ground_pixel  (ground_pixel) int64 2kB 0 1 2 3 4 5 ... 210 211 212 213 214
    lat           (time, scanline, ground_pixel) float32 851MB nan nan ... nan
    lon           (time, scanline, ground_pixel) float32 851MB nan nan ... nan
Data variables:
    deltad        (time, scanline, ground_pixel) float32 851MB nan nan ... nan
    h2o_vmr       (time, scanline, ground_pixel) float32 851MB nan nan ... nan


In [43]:
sys.exit()

SystemExit: 

In [ ]:
print(trp.lat.max().values)
print(trp.lat.min().values)

print(trp.lon.max().values)
print(trp.lon.min().values)

17.345987
-20.417206
39.503304
2.4792442


In [ ]:
#Plot climatology

tpN = trp.where((trp.lat < 12.) & (trp.lat > 5.),drop=True)
tpN = tpN.mean(dim=('scanline','ground_pixel'))
tpN = tpN.groupby('time.dayofyear').mean('time')
#print(tpN['deltad'])
#tpN['deltad'] = tpN['deltad']*1000.0
tpN['deltad'].plot()
plt.title('North TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

tpE = trp.where((trp.lat < 5.) & (trp.lat > -5.),drop=True)
tpE = tpE.mean(dim=('scanline','ground_pixel'))
tpE = tpE.groupby('time.dayofyear').mean('time')
#tpE['deltad'] = tpE['deltad']*1000.0
tpE['deltad'].plot()
plt.title('Equatorial TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

tpS = trp.where((trp.lat < -5.) & (trp.lat > -15.),drop=True)
tpS = tpS.mean(dim=('scanline','ground_pixel'))
tpS = tpS.groupby('time.dayofyear').mean('time')
tpS['deltad'].plot()
plt.title('South TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

KeyboardInterrupt: 